# 🚀 Vertex AI Prompt Optimizer (VAPO) for Radio Transcription

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/optimize_transcription_prompt.ipynb)


This notebook orchestrates the end-to-end server-side Prompt Tuning and Optimization (VAPO) pipeline for your radio transcription models. It compiles your training splits, aligns the dataset schemas, and launches tuning runs on Google's TPU servers.

### 🏁 Pipeline Phases:
1. **Setup & Authentication:** Configure your GCP project variables and authenticate your browser session.
2. **Dataset Compilation:** Reformat raw audio manifests into the `{input_text, target}` schema required by the optimizer.
3. **Optimization Launch:** Submit the server-side tuning job pre-configured with the correct `global` query routing.
4. **Monitoring & Administration:** Check active job states, view logs, and cancel jobs directly from this notebook.

In [ ]:
# @title 1. Install Dependencies & Bootstrap Environment
import sys
import os

# Detect if running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running inside Google Colab. Bootstrapping repository...")
    # Clone repository if not already present
    if not os.path.exists("radio-transcription"):
        !git clone -q https://github.com/watch-duty/radio-transcription.git

    # Install the model library in editable mode
    try:
        import common

        print("✅ Library 'common' already installed.")
    except ImportError:
        print("Installing library and dependencies...")
        # Install in editable mode
        %pip install -q -e radio-transcription/model
        import site
        import importlib

        importlib.reload(site)
        print("\n✅ Library 'common' installed successfully.")

# Install/upgrade the required SDKs and GCS library
%pip install -q --upgrade \
    "google-genai>=2.3,<3" \
    "google-cloud-storage" \
    "pydantic<=2.12.3" \
    tqdm

In [ ]:
# @title 2. Imports
import os
import json
import sys
from google.cloud import storage
from google import genai
from google.genai import types
from google.colab import auth, userdata

# Import repository utilities
from common.gcs_utils import download_jsonl_manifest, upload_text
from common.manifest import is_scoreable_manifest_entry
from common.gemini.prompts import GEMINI_TRANSCRIBE_SYSTEM_PROMPT

In [ ]:
# @title 3. Configure Project Variables

# Retrieve credentials and buckets securely from Colab Secrets (userdata)
# Aligned with the exact key names used in your transcription notebooks!
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET_NAME = userdata.get("GCS_BUCKET")
GCP_PROJECT_NUMBER = userdata.get("GCP_PROJECT_NUMBER")

# Standard defaults if secrets are not set yet in the browser panel
if not GCP_PROJECT_ID:
    GCP_PROJECT_ID = "automatic-hawk-481415-m9"
if not GCS_BUCKET_NAME:
    GCS_BUCKET_NAME = "wd-transcription-data"

GCP_LOCATION = "us-central1"

# @markdown ### Input Configuration
# @markdown Partial path under `gs://{GCS_BUCKET}/segmented_audio/` where `batch_manifest.jsonl` and audio segments are located (e.g., `broadcastify/calls/eval_audio_masked_v2` or `echo/eval_audio_masked_v2`).
INPUT_AUDIO_DIR = (
    "broadcastify/calls/eval_audio_masked_v2"  # @param {type:"string"}
)

# Auto-resolve Project Number via gcloud if not provided in secrets
if not GCP_PROJECT_NUMBER:
    try:
        project_number_raw = subprocess.check_output(
            [
                "gcloud",
                "projects",
                "describe",
                GCP_PROJECT_ID,
                "--format=value(projectNumber)",
            ]
        )
        GCP_PROJECT_NUMBER = project_number_raw.decode("utf-8").strip()
        print(f"✅ Resolved Project Number from GCP: {GCP_PROJECT_NUMBER}")
    except Exception as e:
        print(f"⚠️ Failed to resolve project number automatically: {e}")
        GCP_PROJECT_NUMBER = "781667204380"  # Fallback default
else:
    print(f"✅ Loaded Project Number from Secrets: {GCP_PROJECT_NUMBER}")

In [ ]:
# @title 4. Authenticate with GCP
# Run this cell to authenticate your browser session with Google Cloud
from google.colab import auth

auth.authenticate_user()
print("✅ Browser session authenticated successfully!")

## 🔄 Phase 2: Dataset Reformatting & GCS Upload

The Vertex AI Prompt Optimizer requires the dataset to be in JSONL format, with the input mapped to `input_text` and the human ground-truth mapped **specifically to `"target"`**. This cell reformats your raw manifests and uploads them to GCS.

In [ ]:
# @title Compile and Upload Dataset

# Dynamically construct GCS input and output paths based on your relative INPUT_AUDIO_DIR selection!
gcs_input_manifest = f"gs://{GCS_BUCKET_NAME}/segmented_audio/{INPUT_AUDIO_DIR}/batch_manifest.jsonl"
gcs_output_dataset = f"gs://{GCS_BUCKET_NAME}/segmented_audio/{INPUT_AUDIO_DIR}/apo_dataset.jsonl"

print("=== REFORMATTING DATASET FOR VERTEX AI PROMPT OPTIMIZER ===")
print(f"Source manifest GCS path: {gcs_input_manifest}")
print(f"Target dataset GCS path: {gcs_output_dataset}\n")

# 1. Download manifest in-memory
print("1. Downloading manifest from GCS...")
try:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    manifest_entries = download_jsonl_manifest(
        storage_client, gcs_input_manifest
    )
    print("   ✅ Download successful!")
except Exception as e:
    print(f"   ❌ Failed to download manifest: {e}")
    print(
        "   Please check that INPUT_AUDIO_DIR is correct and contains batch_manifest.jsonl."
    )
    sys.exit(1) if "sys" in sys.modules else None

# 2. Reformat to schema
print("2. Reformatting entries into schema...")
compiled_count = 0
apo_rows = []
try:
    for entry in manifest_entries:
        if not is_scoreable_manifest_entry(entry):
            continue

        audio_uri = entry.get("audio_filepath")
        ground_truth_text = entry.get("text")

        # The VAPO schema maps input to 'input_text' and ground-truth to 'target'!
        apo_entry = {"input_text": audio_uri, "target": ground_truth_text}
        apo_rows.append(json.dumps(apo_entry))
        compiled_count += 1
    print(f"   ✅ Compiled {compiled_count} segments into schema.")
except Exception as e:
    print(f"   ❌ Failed to reformat: {e}")
    sys.exit(1) if "sys" in sys.modules else None

# 3. Upload back to GCS directly from memory
print("3. Uploading compiled dataset back to GCS...")
try:
    jsonl_content = "\n".join(apo_rows) + "\n"
    upload_text(
        storage_client,
        jsonl_content,
        gcs_output_dataset,
        content_type="application/jsonl",
    )
    print(f"   ✅ Upload successful! Dataset is 100% ready for tuning!")
except Exception as e:
    print(f"   ❌ Failed to upload compiled dataset: {e}")

## 🚀 Phase 3: Launch Prompt Optimization Job

Define your baseline jargon-heavy system prompt, construct the schema-compliant VAPO config, and submit the server-side job. 

> ⚠️ **CRITICAL ROUTING SOLUTION:** The target model location is set to **`"global"`** because Google has restricted early release models like `gemini-3.1-flash-lite` to the global endpoints. Setting it to `"global"` satisfies the container's region validation while successfully routing the queries!

In [ ]:
# @title Edit Prompt and Submit Job

# @markdown ### Define the Baseline System Prompt to Optimize
# To experiment with a different system prompt, assign it as a string to `custom_system_prompt` below.
# If left as None or empty, it defaults to the production baseline `GEMINI_TRANSCRIBE_SYSTEM_PROMPT`.
custom_system_prompt = None

baseline_prompt = custom_system_prompt or GEMINI_TRANSCRIBE_SYSTEM_PROMPT

# Dynamically link training dataset and output prefix based on your relative paths!
train_dataset_uri = f"gs://{GCS_BUCKET_NAME}/segmented_audio/{INPUT_AUDIO_DIR}/apo_dataset.jsonl"
dynamic_output_prefix = f"segmented_audio/{INPUT_AUDIO_DIR}/vapo_outputs"

# 1. Construct VAPO configuration
vapo_data_settings = {
    "system_instruction": baseline_prompt,
    # 🎥 Mark {input_text} as multimodal GCS audio URI
    "prompt_template": "{input_text} @@@audio/flac\n{target}",
    "target_model": "gemini-3.1-flash-lite",  # Target production model
    "thinking_budget": 0,
    "optimization_mode": "instruction_and_demo",  # Optimize instruction + few-shot examples
    "eval_metrics_types": ["bleu", "rouge_l"],
    "eval_metrics_weights": [0.5, 0.5],
    "input_data_path": train_dataset_uri,
    "output_path": f"gs://{GCS_BUCKET_NAME}/{dynamic_output_prefix}/results/",
    "project": GCP_PROJECT_ID,
    "num_steps": 10,
    "num_demo_set_candidates": 10,
    "demo_set_size": 3,
    # 🔗 Global Routing Solution
    "target_model_location": "global",
    "optimizer_model_location": "global",
    "has_multimodal_inputs": True,
    "data_limit": 50,
}

# 2. Upload config.json to GCS
config_gcs_path = f"gs://{GCS_BUCKET_NAME}/{dynamic_output_prefix}/config.json"
print(f"1. Uploading JSON configuration to GCS: {config_gcs_path}...")
storage_client = storage.Client(project=GCP_PROJECT_ID)
upload_text(
    storage_client,
    json.dumps(vapo_data_settings, indent=2),
    config_gcs_path,
    content_type="application/json",
)
print("   ✅ Configuration uploaded successfully!")

# 3. Initialize Vertex AI client using the unified google-genai SDK
print(f"\n2. Initializing Vertex AI Client in {GCP_LOCATION}...")
client = genai.Client(
    vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION
)

# 4. Launch the optimization job
print(f"\n3. Submitting Server-Side Prompt Optimization Job...")
service_account = f"{GCP_PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
vapo_run_config = {
    "config_path": config_gcs_path,
    "wait_for_completion": False,  # Submit and return immediately
    "service_account": service_account,
}

try:
    # Suppress experimental warning
    import warnings

    warnings.filterwarnings("ignore", category=UserWarning)

    result = client.prompts.launch_optimization_job(
        method=types.PromptOptimizerMethod.VAPO, config=vapo_run_config
    )
    print("\n==================================================")
    print("🎉 SUCCESS! PROMPT OPTIMIZER JOB SUBMITTED!")
    print("==================================================")
    print(f"Directory: {INPUT_AUDIO_DIR}")
    print(f"Service Account: {service_account}")
    print(f"Check your Cloud Console Training -> Custom Jobs list shortly!")
    print("==================================================\n")
except Exception as e:
    print(f"  ❌ Failed to submit job: {e}")

## 🕵️‍♂️ Phase 4: Monitoring and Job Administration

Use these cells to query the queue, check specific job states, or cancel unwanted runs directly from the notebook.

In [ ]:
# @title 1. List Recent Custom Jobs
# Query the 4 most recent Custom Jobs in us-central1
!gcloud ai custom-jobs list \
    --project={GCP_PROJECT_ID} \
    --region={GCP_LOCATION} \
    --limit=4 \
    --format="table(name.basename():label=JOB_ID, state:label=STATUS, createTime:label=CREATED_TIME)"

In [ ]:
# @title 2. Describe Specific Job State
TARGET_JOB_ID = ""  # @param {type:"string"}

if TARGET_JOB_ID:
    !gcloud ai custom-jobs describe {TARGET_JOB_ID} \
        --project={GCP_PROJECT_ID} \
        --region={GCP_LOCATION} \
        --format="value(state)"
else:
    print("⚠️ Please enter a TARGET_JOB_ID in the form parameter above!")

In [ ]:
# @title 3. Cancel Running Custom Job
CANCEL_JOB_ID = ""  # @param {type:"string"}

if CANCEL_JOB_ID:
    confirm = input(
        f"Are you absolutely sure you want to CANCEL job {CANCEL_JOB_ID}? (y/n): "
    )
    if confirm.lower() == "y":
        !gcloud ai custom-jobs cancel {CANCEL_JOB_ID} \
            --project={GCP_PROJECT_ID} \
            --region={GCP_LOCATION} \
            --quiet
        print(f"✅ Cancel request sent for job {CANCEL_JOB_ID}!")
    else:
        print("Cancellation aborted.")
else:
    print("⚠️ Please enter a CANCEL_JOB_ID in the form parameter above!")

In [ ]:
# @title 4. Print the Final Optimized System Instruction
# Run this cell to instantly read GCS and print the final winning prompt!
import json
from etils import epath

# Construct the path to the final optimized results file based on your relative path
optimized_results_path = f"gs://{GCS_BUCKET_NAME}/segmented_audio/{INPUT_AUDIO_DIR}/vapo_outputs/results/instruction/demonstration/optimized_results.json"
print(f"Reading final optimized results from: {optimized_results_path}\n")

try:
    with epath.Path(optimized_results_path).open("r") as f:
        data = json.load(f)

    winning_prompt = data.get("prompt", "")
    print("==================================================")
    print("🏆 WINNING SYSTEM INSTRUCTION:")
    print("==================================================")
    print(winning_prompt)
    print("==================================================\n")
except Exception as e:
    print(f"❌ Failed to load optimized prompt: {e}")
    print("Ensure the job has completed and the path is correct.")

## 📊 Phase 5: Visualize Results with Interactive Gradio App

This section launches a **Gradio-based interactive web application** directly from your notebook. It lets you visually explore and compare the results of different prompt optimization runs, view the mutated prompts, and analyze segment-level BLEU/ROUGE improvements.

In [ ]:
# @title 1. Build the Gradio Application
import io
import re
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd
from etils import epath
import gradio as gr
import json
from google.cloud import storage


def find_vapo_runs(base_path: str) -> List[str]:
    """Find VAPO run directories containing required files."""
    REQUIRED_FILES = {"eval_results.json", "templates.json"}
    bucket_name, prefix = parse_gcs_path(base_path)
    all_objects = list_gcs_objects(base_path)

    # Group files by directory
    directories = {}
    for obj_path in all_objects:
        dir_path = "/".join(obj_path.split("/")[:-1])
        filename = obj_path.split("/")[-1]

        if dir_path not in directories:
            directories[dir_path] = set()
        directories[dir_path].add(filename)

    # Find directories with all required files
    valid_runs = []
    for dir_path, files in directories.items():
        if REQUIRED_FILES.issubset(files):
            valid_runs.append(f"gs://{bucket_name}/{dir_path}")

    return valid_runs


def extract_metric_name(column_name: str) -> str:
    """Extract clean metric name from column name."""
    match = re.search(r"\.(\w+)/", column_name)
    if match:
        return match.group(1)
    parts = column_name.split(".")
    if parts:
        return parts[-1].split("/")[0]
    return column_name


def is_metric_column(col: str) -> bool:
    """Check if a column is a metric column with score, confidence, or explanation."""
    return bool(re.search(r"[\/\.](score|confidence|explanation)$", col))


def is_core_field(col: str) -> bool:
    """Check if a column is a core field (not a metric)."""
    CORE_FIELDS = [
        "question",
        "target",
        "ctx",
        "context",
        "prompt",
        "response",
        "reference",
    ]
    col_lower = col.lower()
    if is_metric_column(col):
        return False
    return col_lower in CORE_FIELDS


def is_duplicate_text_field(col: str, all_columns: List[str]) -> bool:
    """Check if a column is a duplicate text field that should be excluded."""
    if is_metric_column(col):
        return False
    CORE_FIELDS = [
        "question",
        "target",
        "ctx",
        "context",
        "prompt",
        "response",
        "reference",
    ]
    col_lower = col.lower()
    for core_field in CORE_FIELDS:
        if (
            col_lower.startswith(f"{core_field}_")
            or f"_{core_field}" in col_lower
        ):
            return col_lower != core_field

    metric_names = ["fluency", "correctness", "relevance", "coherence"]
    for metric_name in metric_names:
        if col_lower.endswith(f"_{metric_name}") or col_lower == metric_name:
            has_metric_version = any(
                metric_name in c.lower() and is_metric_column(c)
                for c in all_columns
            )
            if has_metric_version:
                return True
    return False


def filter_columns_for_display(columns: List[str]) -> List[str]:
    """Filter columns to keep only core fields and metric score/confidence/explanations."""
    filtered = []
    seen_context = False
    for col in columns:
        col_lower = col.lower()
        if is_duplicate_text_field(col, columns):
            continue
        if is_core_field(col):
            if col_lower == "context":
                if not seen_context and "ctx" not in [
                    c.lower() for c in filtered
                ]:
                    filtered.append(col)
                    seen_context = True
            elif col_lower == "ctx":
                filtered = [c for c in filtered if c.lower() != "context"]
                filtered.append(col)
                seen_context = True
            else:
                filtered.append(col)
        elif is_metric_column(col):
            filtered.append(col)
    return filtered


def process_evaluation_result(result: Dict[str, Any]) -> pd.DataFrame:
    """Process evaluation result for clean display."""
    df = pd.read_json(io.StringIO(result["metrics_table"]))
    return df


def categorize_columns(columns: List[str]) -> Dict[str, List[str]]:
    """Categorize columns by type for better organization."""
    categories = {
        "Core Fields": [],
        "Metric Scores": [],
        "Metric Confidence": [],
        "Metric Explanations": [],
    }
    for col in columns:
        if is_core_field(col):
            categories["Core Fields"].append(col)
        elif col.endswith("/score") or col.endswith(".score"):
            categories["Metric Scores"].append(col)
        elif col.endswith("/confidence") or col.endswith(".confidence"):
            categories["Metric Confidence"].append(col)
        elif col.endswith("/explanation") or col.endswith(".explanation"):
            categories["Metric Explanations"].append(col)
    return {k: v for k, v in categories.items() if v}


def get_default_columns(all_columns: List[str]) -> List[str]:
    """Get default columns to display - only the specified core fields."""
    DEFAULT_FIELDS = [
        "question",
        "target",
        "ctx",
        "prompt",
        "response",
        "reference",
    ]
    default_cols = []
    for field in DEFAULT_FIELDS:
        for col in all_columns:
            if col.lower() == field and col not in default_cols:
                default_cols.append(col)
                break
    if not any(col.lower() == "ctx" for col in default_cols):
        for col in all_columns:
            if col.lower() == "context" and col not in default_cols:
                default_cols.append(col)
                break
    return default_cols


def simplify_dataframe_for_metrics(
    df: pd.DataFrame, selected_columns: List[str]
) -> pd.DataFrame:
    """Simplify DataFrame by filtering and formatting metric columns."""
    if df.empty or not selected_columns:
        return df
    valid_columns = [col for col in selected_columns if col in df.columns]
    if not valid_columns:
        return pd.DataFrame()
    simplified_df = df[valid_columns].copy()
    for col in simplified_df.columns:
        if simplified_df[col].dtype in ["float64", "float32"]:
            if any(
                x in col
                for x in ["/score", ".score", "/confidence", ".confidence"]
            ):
                simplified_df[col] = simplified_df[col].round(3)
    for col in simplified_df.columns:
        if "/explanation" in col or ".explanation" in col:
            simplified_df[col] = simplified_df[col].apply(
                lambda x: (
                    x[:200] + "..."
                    if isinstance(x, str) and len(x) > 200
                    else x
                )
            )
    return simplified_df


def parse_gcs_path(gcs_path: str) -> Tuple[str, str]:
    """Parse GCS path into bucket name and prefix."""
    if not gcs_path.startswith("gs://"):
        raise ValueError("Invalid GCS path.")
    path_without_prefix = gcs_path[5:]
    parts = path_without_prefix.split("/", 1)
    return parts[0], parts[1] if len(parts) > 1 else ""


def list_gcs_objects(gcs_path: str) -> List[str]:
    """List all objects under given GCS path."""
    bucket_name, prefix = parse_gcs_path(gcs_path)
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)
    return [blob.name for blob in blobs]


class VAPOResultsViewer:
    """Main application class for VAPO results viewer."""

    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.base_path = None
        self.runs = []
        self.templates = []
        self.eval_results = []
        self.current_run = None
        self.current_eval_full = None
        self.available_columns = []
        self.filtered_columns = []

    def load_runs(self, base_path: str) -> gr.Dropdown:
        if not base_path:
            return gr.Dropdown(choices=[], value=None)
        try:
            self.base_path = base_path
            self.runs = find_vapo_runs(base_path)
            if not self.runs:
                return gr.Dropdown(choices=[], value=None)
            return gr.Dropdown(choices=self.runs, value=self.runs[0])
        except Exception as e:
            print(f"Error loading runs: {e}")
            return gr.Dropdown(choices=[], value=None)

    def load_run_data(
        self, run_path: str
    ) -> Tuple[gr.Dropdown, pd.DataFrame, pd.DataFrame, gr.CheckboxGroup]:
        if not run_path:
            return (
                gr.Dropdown(choices=[], value=None),
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )
        try:
            self.current_run = run_path
            templates_data = self._load_json(f"{run_path}/templates.json")
            eval_data = self._load_json(f"{run_path}/eval_results.json")
            self.templates = [pd.json_normalize(t) for t in templates_data]
            self.eval_results = [
                process_evaluation_result(r) for r in eval_data
            ]

            if len(self.templates) == len(self.eval_results) + 1:
                self.templates = self.templates[1:]
            elif len(self.templates) != len(self.eval_results):
                raise ValueError("Mismatch in templates vs results length.")

            template_options = self._create_template_options()
            if template_options:
                template_df, eval_df, column_selector = self._get_template_data(
                    0
                )
                return (
                    gr.Dropdown(
                        choices=template_options, value=template_options[0]
                    ),
                    template_df,
                    eval_df,
                    column_selector,
                )
            return (
                gr.Dropdown(choices=[], value=None),
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )
        except Exception as e:
            print(f"Error loading run data: {e}")
            return (
                gr.Dropdown(choices=[], value=None),
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )

    def _load_json(self, path: str) -> Any:
        if path.startswith("gs://"):
            bucket_name, prefix = parse_gcs_path(path)
            client = storage.Client(project=GCP_PROJECT_ID)
            bucket = client.bucket(bucket_name)
            blob = bucket.blob(prefix)
            content = blob.download_as_text()
            return json.loads(content)
        else:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)

    def _create_template_options(self) -> List[str]:
        options = []
        for i, template_df in enumerate(self.templates):
            metrics = []
            for col in template_df.columns:
                if "metric" in col and "mean" in col:
                    value = template_df[col].iloc[0]
                    metric_name = extract_metric_name(col)
                    metrics.append(f"{metric_name}: {value:.3f}")
            metrics_str = " | ".join(metrics) if metrics else "No metrics"
            options.append(f"Template {i} - {metrics_str}")
        return options

    def _get_template_data(
        self, index: int
    ) -> Tuple[pd.DataFrame, pd.DataFrame, gr.CheckboxGroup]:
        if not (0 <= index < len(self.templates)):
            return (
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )
        template_df = self.templates[index].T.reset_index()
        template_df.columns = ["Field", "Value"]
        self.current_eval_full = self.eval_results[index]
        self.available_columns = list(self.current_eval_full.columns)
        self.filtered_columns = filter_columns_for_display(
            self.available_columns
        )
        default_columns = get_default_columns(self.filtered_columns)
        eval_df = simplify_dataframe_for_metrics(
            self.current_eval_full, default_columns
        )
        column_selector = gr.CheckboxGroup(
            choices=self.filtered_columns,
            value=default_columns,
            label="Select Columns to Display",
        )
        return template_df, eval_df, column_selector

    def display_template(
        self, template_selection: str
    ) -> Tuple[pd.DataFrame, pd.DataFrame, gr.CheckboxGroup]:
        if not template_selection:
            return (
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )
        try:
            index = int(template_selection.split()[1])
            return self._get_template_data(index)
        except Exception:
            return (
                pd.DataFrame(),
                pd.DataFrame(),
                gr.CheckboxGroup(choices=[], value=[]),
            )

    def update_evaluation_display(
        self, selected_columns: List[str]
    ) -> pd.DataFrame:
        if not selected_columns or self.current_eval_full is None:
            return pd.DataFrame()
        return simplify_dataframe_for_metrics(
            self.current_eval_full, selected_columns
        )

    def create_interface(self, default_path: str = "") -> gr.Blocks:
        with gr.Blocks(title="VAPO Results Viewer") as interface:
            gr.Markdown("# 🚀 VAPO Results Viewer")
            gr.Markdown(
                "Explore mutated system instructions and segment-level BLEU/ROUGE score improvements."
            )
            with gr.Row():
                with gr.Column(scale=3):
                    self.base_path_input = gr.Textbox(
                        label="GCS Base Path",
                        value=default_path,
                        placeholder="gs://your-bucket/vapo-results",
                    )
                with gr.Column(scale=1):
                    load_btn = gr.Button("Load Runs", variant="primary")
            with gr.Row():
                self.run_dropdown = gr.Dropdown(
                    label="Select Run", choices=[], interactive=True
                )
                self.template_dropdown = gr.Dropdown(
                    label="Select Template/Iteration",
                    choices=[],
                    interactive=True,
                )
            with gr.Tabs():
                with gr.Tab("Mutated System Instruction"):
                    self.template_display = gr.DataFrame(
                        label="Template Details", wrap=True, interactive=False
                    )
                with gr.Tab("Segment-level Evaluation Metrics"):
                    with gr.Accordion("📊 Column Selection", open=True):
                        self.column_selector = gr.CheckboxGroup(
                            choices=[], value=[], label="Select Columns"
                        )
                    self.eval_display = gr.DataFrame(
                        label="Evaluation Table",
                        wrap=True,
                        interactive=False,
                        row_count=20,
                    )
            load_btn.click(
                fn=self.load_runs,
                inputs=[self.base_path_input],
                outputs=[self.run_dropdown],
            )
            self.run_dropdown.change(
                fn=self.load_run_data,
                inputs=[self.run_dropdown],
                outputs=[
                    self.template_dropdown,
                    self.template_display,
                    self.eval_display,
                    self.column_selector,
                ],
            )
            self.template_dropdown.change(
                fn=self.display_template,
                inputs=[self.template_dropdown],
                outputs=[
                    self.template_display,
                    self.eval_display,
                    self.column_selector,
                ],
            )
            self.column_selector.change(
                fn=self.update_evaluation_display,
                inputs=[self.column_selector],
                outputs=[self.eval_display],
            )
        return interface

In [ ]:
# @title 2. Launch VAPO Results Viewer
# Pre-populate the GCS base path input to point directly to your current relative path's results!
default_gcs_path = f"gs://{GCS_BUCKET_NAME}/segmented_audio/{INPUT_AUDIO_DIR}/vapo_outputs/results"
print(f"🔗 Default GCS Results Path: {default_gcs_path}")

# Instantiate and launch
viewer = VAPOResultsViewer()
interface = viewer.create_interface(default_path=default_gcs_path)

# Use Colab's native secure port-forwarding proxy to bypass corporate firewalls!
interface.launch(
    share=False,  # 🌟 Disable gradio.live reverse tunnel to bypass firewalls
    inline=True,  # 🌟 Render securely inside the Colab cell output
    server_port=7861,
    server_name="0.0.0.0",
    debug=True,
)